# 03b. Regularization

**Goals for this notebook**

- See two situations where plain logistic regression **goes wrong**:
  - data that is *too easy* (perfectly separable), where the weights grow forever;
  - many features and few examples, where the model memorizes the training data.
- Understand the fix, **regularization**: a penalty on large weights, and what it does to the gradient.
- Finally understand sklearn's parameter **`C`**, the "extra term" sklearn has been adding since 01b.
- See what different values of `C` do, in pictures and in numbers, and learn how to **choose** `C`
  properly, **without touching the test set**.

## 0. Setup

Besides the usual imports, the second cell has **our** implementation of logistic regression from
03a. (You can paste in yours instead: it's the same thing.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

In [ ]:
# ---- our implementation from 03a ----
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)

def log_loss(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def predict(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyJXMSI6IDMsICJXMiI6ICJiIiwgIlczIjogMiwgIjEuQSI6IFswLjA2ODc1MzEwMjM1NTg4Mjk5LCAwLjAxMzAyODQxMTk4MjUzMTgxMywgMC4wMDEyNDkzMDk0NDIwNjAzNTQzLCAyLjQ4NzczNjYwMDIyMjY5NzVlLTA1XSwgIjIuQSI6IFsxLjAsIDAuODkwMzU5MTY4MjQxOTY1OV0sICIzLkEiOiBbMC4xOTk4OTQwNzkzMjU2ODg3NiwgMC4yOTU0Mjg5NTgxNDcxMjE2NywgLTAuMzQxMTQwMDk1NzkwNTY3MywgMC42MTc0NzE2MjE3NzM2NDA4XSwgIjMuQiI6IFsyLjY3NTAxMjU4ODcxNzI0MTMsIDEuMzc3MjQwMTIwMzM1NDNdLCAiNC5BIjogMC4wMDM1MTQ5Mzg0ODg1NzY0NDk3LCAiNC5CIjogImEiLCAiNS5BIjogIndvcnN0IHJhZGl1cyJ9"))

def _has_str(v):
    return isinstance(v, str) or (isinstance(v, (list, tuple)) and any(_has_str(x) for x in v))

def _norm(v):
    return [_norm(x) for x in v] if isinstance(v, (list, tuple, np.ndarray)) else str(v).strip().lower()

def _close(a, b, tol):
    if _has_str(b):
        return _norm(a) == _norm(b)
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_against(key, value, reference, tol=1e-3):
    # Compare your answer with a reference value computed in the same cell.
    if value is None or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, reference, tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

## Warm-up: notebook 03a in three questions

**W1.** `X` has shape `(100, 3)` and `err` has shape `(100,)`. What is the length of `X.T @ err`?
Store it in `w1_answer`.

**W2.** During gradient descent, the loss **jumps up and down** instead of decreasing smoothly. What's
the most likely cause? Store the letter in `w2_answer`.
- (a) the learning rate is too small
- (b) the learning rate is too large
- (c) the data isn't scaled, so the model can't learn anything

**W3.** In 03a's optional challenge, $\tfrac12(\tanh(z) + 1) = \sigma(k z)$ for which number $k$?
Store it in `w3_answer`.

In [ ]:
# YOUR CODE HERE
w1_answer = None
w2_answer = None
w3_answer = None
check("W1", w1_answer)
check("W2", w2_answer)
check("W3", w3_answer)

## 1. Problem 1: data that is too easy

Here's a small 2D dataset in which the two classes can be separated **perfectly** by a straight line.
Such data is called **linearly separable**.

In [ ]:
rng = np.random.default_rng(3)
X_sep = np.vstack([rng.normal([-1.5, -1.0], 0.6, size=(30, 2)),
                   rng.normal([1.5, 1.0], 0.6, size=(30, 2))])
y_sep = np.r_[np.zeros(30), np.ones(30)].astype(int)

plt.scatter(X_sep[:, 0], X_sep[:, 1], c=y_sep, cmap="coolwarm", edgecolor="k")
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Linearly separable data")
plt.show()

Let's train logistic regression on it with gradient descent, for a long time, and track two things
at every step: the loss, and the **length of the weight vector**, $\|w\| = \sqrt{w_1^2 + w_2^2}$.

In [ ]:
def fit_tracking(X, y, eta, n_iter):
    # gradient descent (as in 03a), also recording ||w|| after every step
    w, b = np.zeros(X.shape[1]), 0.0
    losses, norms = [], []
    for _ in range(n_iter):
        h = predict_proba(X, w, b)
        losses.append(log_loss(y, h))
        w = w - eta * X.T @ (h - y) / len(y)
        b = b - eta * np.mean(h - y)
        norms.append(np.linalg.norm(w))
    return w, b, losses, norms

w_sep, b_sep, losses_sep, norms_sep = fit_tracking(X_sep, y_sep, eta=1.0, n_iter=20000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses_sep); axes[0].set_yscale("log")
axes[0].set_title("log-loss (log scale)"); axes[0].set_xlabel("step")
axes[1].plot(norms_sep); axes[1].set_xscale("log")
axes[1].set_title("||w||, the length of w"); axes[1].set_xlabel("step (log scale)")
plt.show()
for k in [10, 100, 1000, 10000, 20000]:
    print(f"after {k:>5} steps: loss = {losses_sep[k - 1]:.5f},  ||w|| = {norms_sep[k - 1]:.2f}")

The loss keeps going **down toward 0**, and $\|w\|$ keeps going **up**, with no sign of stopping.
(On the log-scale plot it's a straight line going up: $\|w\|$ grows roughly like the logarithm of the
number of steps. Slowly, but forever.)

What does a huge $\|w\|$ mean? Remember from 01b: multiplying $w$ and $b$ by a constant keeps the
boundary in place but makes the transition **sharper**. Let's look at the model at a few moments
during training.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
g1, g2 = np.meshgrid(np.linspace(-3.5, 3.5, 200), np.linspace(-3, 3, 200))
grid = np.column_stack([g1.ravel(), g2.ravel()])
for ax, steps in zip(axes, [10, 100, 1000, 20000]):
    w_k, b_k, _, _ = fit_tracking(X_sep, y_sep, eta=1.0, n_iter=steps)
    ax.contourf(g1, g2, predict_proba(grid, w_k, b_k).reshape(g1.shape),
                levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.75)
    ax.scatter(X_sep[:, 0], X_sep[:, 1], c=y_sep, cmap="coolwarm", edgecolor="k", s=20)
    ax.set_title(f"after {steps} steps, ||w|| = {np.linalg.norm(w_k):.1f}")
plt.show()

The band between the colours gets **narrower and narrower**: the probability map is heading toward a
**step**, where the model is **100% certain** about everything, including about points near the
boundary, where no reasonable person would be certain. And it never stops: the longer we train, the
more extreme it gets.

**Why does this happen?** If some line separates the data perfectly, then every example is on its
correct side, so every score $w^T x_i + b$ has the right sign. Now double $w$ and $b$: the line stays
the same, but every score doubles, so every probability moves **closer** to the correct label, and the
log-loss **decreases**. Double again: it decreases again. There is always a better $w$, just
further away. **The log-loss has no minimum**, and gradient descent walks toward infinity forever.

### Exercise 1.A: seeing it in numbers

The direction $w = (1.5, 1)$ with $b = 0$ separates the data perfectly. Compute the log-loss of the
models $k \cdot w$ (with $b = 0$) for $k = 1, 2, 4, 8$. Store the four losses in the list
`losses_scaled`. Is it always decreasing?

In [ ]:
w_dir = np.array([1.5, 1.0])
# YOUR CODE HERE
losses_scaled = None
print(losses_scaled)
check("1.A", losses_scaled)

### Exercise 1.B (in words)

1. Why is a model that's "100% sure" a bad thing, even if it classifies all the training points
   correctly?
2. Would this problem happen if the two classes **overlapped**, as in the 01b clinic data? Why or why
   not?

**Your answer:** *(write here)*

## 2. Problem 2: many features, few examples

The second problem is closely related. Take the Breast Cancer data from 02 (30 features) but, to make
the problem obvious, train on only **40** patients. With 30 features and 40 examples, there's a lot of
freedom: the model can find weights that fit the **training** patients perfectly, including their
noise and quirks.

In [ ]:
cancer = load_breast_cancer()
X_bc, y_bc = cancer.data, 1 - cancer.target              # 1 = malignant (as in 02)
bc_names = list(cancer.feature_names)
X_tr_small, X_te_small, y_tr_small, y_te_small = train_test_split(
    X_bc, y_bc, train_size=40, stratify=y_bc, random_state=0)   # only 40 training patients!
sc_small = StandardScaler().fit(X_tr_small)
X_tr_small, X_te_small = sc_small.transform(X_tr_small), sc_small.transform(X_te_small)

print("training:", X_tr_small.shape, "   test:", X_te_small.shape)

### Exercise 2.A

Fit a `LogisticRegression` with `C=1e10` (practically no regularization, as in 03a) and `max_iter=10000`
on the 40 training patients. Compute its **training** accuracy and its **test** accuracy, and store
them in `acc_2a = [train_accuracy, test_accuracy]`. Then print the length of its weight vector.

In [ ]:
# YOUR CODE HERE
acc_2a = None
print("train accuracy, test accuracy:", acc_2a)
check("2.A", acc_2a, tol=0.02)

**Your answer:** *(write here)*

## 3. The fix: penalize large weights

Both problems come from weights that grow too large. The fix is simple: **make large weights cost
something.** We add a penalty to the loss:

$$L_{\text{reg}}(w, b) = \underbrace{-\frac{1}{N}\sum_{i=1}^N \big[y_i \log h_i + (1-y_i)\log(1-h_i)\big]}_{\text{log-loss: fit the data}}
\;+\; \underbrace{\frac{\lambda}{2}\,\|w\|^2}_{\text{penalty: keep } w \text{ small}}$$

where $\|w\|^2 = w_1^2 + \dots + w_n^2$ and $\lambda \ge 0$ (the Greek letter *lambda*) is the
**regularization strength**. Some points:

- Now there's a **tug of war**: the log-loss wants $w$ large (on separable data, infinitely large),
  and the penalty wants $w$ small. The minimum is a compromise, at a **finite** $w$.
- $\lambda = 0$ is plain logistic regression. A larger $\lambda$ means a stronger pull toward small
  weights.
- The intercept $b$ is **not** penalized: it only shifts the boundary, and doesn't make the model more
  "extreme".
- This particular penalty is called **L2 regularization** (after the "L2 norm" $\|w\|$), or
  *ridge*.

**The gradient.** The derivative of $\frac{\lambda}{2}\|w\|^2 = \frac{\lambda}{2}\sum_j w_j^2$ with
respect to $w_j$ is simply $\lambda w_j$. So:

$$\nabla_w L_{\text{reg}} = \frac{1}{N}\sum_{i} (h_i - y_i)\,x_i \;+\; \lambda w, \qquad
\frac{\partial L_{\text{reg}}}{\partial b} = \frac{1}{N}\sum_i (h_i - y_i) \quad \text{(unchanged)}$$

and a gradient-descent step becomes
$$w \leftarrow w - \eta\Big(\tfrac{1}{N}\textstyle\sum_i (h_i - y_i)\,x_i\Big) - \eta\lambda w$$

The new term $-\eta\lambda w$ shrinks $w$ a little toward zero at **every** step. This is why L2
regularization is also called **weight decay**.

### Exercise 3.A: add the penalty

Write `fit_reg(X, y, eta, n_iter, lam)`: like `fit_tracking` above, but with the penalty.

- The gradient for $w$ gets the extra term `lam * w`.
- Record the **regularized** loss (log-loss plus $\frac{\lambda}{2}\|w\|^2$).
- Return `w, b, losses, norms`.

In [ ]:
def fit_reg(X, y, eta, n_iter, lam=0.0):
    # YOUR CODE HERE
    return None, None, None, None

X_small = np.array([[1.0, 2.0], [2.0, 0.0], [0.0, 1.0], [3.0, 1.0]])
y_small = np.array([1, 0, 0, 1])
w3, b3, L3, n3 = fit_reg(X_small, y_small, eta=0.1, n_iter=50, lam=0.5)
check("3.A", None if w3 is None else list(w3) + [b3, L3[-1]])

def reg_done():
    try:
        return fit_reg(X_small, y_small, 0.1, 1, 0.1)[0] is not None
    except Exception:
        return False

### Exercise 3.B: the runaway weights, tamed

Train on the separable data again (with `eta=1.0`, 5000 steps), with $\lambda = 0.01$ and with
$\lambda = 0.1$. Store the **final** $\|w\|$ of each in `norms_3b = [norm_001, norm_01]`. The plotting
cell then compares the three curves ($\lambda = 0$ from section 1, and your two).

In [ ]:
# YOUR CODE HERE
norms_3b = None
print(norms_3b)
check("3.B", norms_3b)

In [ ]:
if reg_done():
    plt.figure(figsize=(7, 4))
    plt.plot(norms_sep[:5000], label="lambda = 0 (no penalty)")
    for lam in [0.01, 0.1]:
        _, _, _, nr = fit_reg(X_sep, y_sep, eta=1.0, n_iter=5000, lam=lam)
        plt.plot(nr, label=f"lambda = {lam}")
    plt.xscale("log"); plt.xlabel("step (log scale)"); plt.ylabel("||w||")
    plt.legend(); plt.title("With a penalty, ||w|| levels off")
    plt.show()
else:
    print("⏳ finish exercise 3.A first.")

With the penalty, $\|w\|$ **levels off**: the loss now has a real minimum, and a larger $\lambda$
gives a smaller $w$, which means a **softer** probability map, a less confident model.

## 4. sklearn's `C`

Now we can finally decode sklearn's `C`. sklearn writes the regularized loss a little differently:

$$\text{sklearn minimizes:}\qquad C \cdot \sum_{i=1}^N \ell_i \;+\; \frac{1}{2}\|w\|^2$$

(a **sum** of the per-example log-losses $\ell_i$, not an average, and the constant $C$ in front of the
data term instead of in front of the penalty). Dividing everything by $C \cdot N$ doesn't change where
the minimum is, and gives:

$$\frac{1}{N}\sum_{i=1}^N \ell_i \;+\; \frac{1}{2 C N}\|w\|^2$$

This is exactly our $L_{\text{reg}}$ with $\lambda = \dfrac{1}{C \cdot N}$. (The current sklearn
documentation writes the formula in this second, already-divided form. It's the same thing.) So:

- **`C` is the *inverse* of the regularization strength.** A **small** `C` means **strong**
  regularization, and a **large** `C` means weak regularization.
- `C=1e10` makes $\lambda$ practically 0: that's why we used it in 03a to turn regularization off.
- sklearn's **default** is `C=1.0`: a mild regularization that has been quietly present since 01b.
  That's why the banknote model in 02 was more cautious than it needed to be.

### Exercise 4.A: the same model, two ways

1. The full Breast Cancer dataset has $N = 569$ patients. Which $\lambda$ corresponds to `C=0.5`? Store it
   in `lam_4a`.
2. The next cell fits both your `fit_reg` (with that $\lambda$) and sklearn's `LogisticRegression(C=0.5)`
   on the scaled data, and compares their weights. They should agree closely.

In [ ]:
# YOUR CODE HERE
lam_4a = None
check("4.A", lam_4a, tol=1e-6)

In [ ]:
if (lambda: reg_done() and lam_4a is not None)():
    X_bc_s = StandardScaler().fit_transform(X_bc)
    w_mine, b_mine, _, _ = fit_reg(X_bc_s, y_bc, eta=0.5, n_iter=20000, lam=lam_4a)
    sk = LogisticRegression(C=0.5).fit(X_bc_s, y_bc)
    print("largest difference between the 30 weights:", np.abs(w_mine - sk.coef_[0]).max().round(4))
    print("difference between the intercepts:        ", abs(b_mine - sk.intercept_[0]).round(4))
else:
    print("⏳ finish exercises 3.A and 4.A first.")

The weights agree to about two decimal places. The small remaining difference is only because
our gradient descent, after 20000 steps, hasn't *quite* reached the exact minimum that sklearn's
smarter optimizer finds.

### Exercise 4.B

Which of these models is **more** regularized, (a) `LogisticRegression(C=0.01)` or
(b) `LogisticRegression(C=100)`? Store the letter in `answer_4b`. Which one will have the **larger**
weights?

In [ ]:
# YOUR CODE HERE
answer_4b = None
check("4.B", answer_4b)

## 5. What different values of `C` do

We'll look at the effect of `C` from three angles, for
$C \in \{0.001, 0.01, 0.1, 1, 10, 100, 1000\}$: pictures, weights, and performance.

### 5.1 In pictures

The 01b clinic data (scaled), fitted with different values of `C`:

In [ ]:
def make_patients(n, seed):
    rng = np.random.default_rng(seed)
    y = (rng.uniform(size=n) < 0.4).astype(int)
    healthy = rng.normal([3.5, 3.0], 1.3, size=(n, 2))
    sick = rng.normal([6.0, 5.0], 1.3, size=(n, 2))
    return np.round(np.where(y[:, None] == 1, sick, healthy), 2), y

X_pat, y_pat = make_patients(200, seed=0)
X_pat_s = StandardScaler().fit_transform(X_pat)
g1, g2 = np.meshgrid(np.linspace(-3, 3.2, 200), np.linspace(-4, 3.2, 200))
grid = np.column_stack([g1.ravel(), g2.ravel()])

def plot_C(C, ax=None):
    ax = ax or plt.gca()
    m = LogisticRegression(C=C).fit(X_pat_s, y_pat)
    probs = m.predict_proba(grid)[:, 1].reshape(g1.shape)
    ax.contourf(g1, g2, probs, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.75)
    ax.contour(g1, g2, probs, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X_pat_s[:, 0], X_pat_s[:, 1], c=y_pat, cmap="coolwarm", edgecolor="k", s=12)
    ax.set_title(f"C = {C:g},  ||w|| = {np.linalg.norm(m.coef_[0]):.2f}")

if HAVE_WIDGETS:
    def show_C(log10_C=0.0):
        plt.figure(figsize=(5.5, 5)); plot_C(10 ** log10_C); plt.show()
    interact(show_C, log10_C=FloatSlider(value=0, min=-3, max=3, step=0.25, description="log10(C)"))
else:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
    for ax, C in zip(axes.ravel(), [0.001, 0.01, 0.1, 1, 10, 100, 1000]):
        plot_C(C, ax)
    axes.ravel()[-1].axis("off")
    plt.show()

- With a **tiny** `C` (strong regularization), $w$ is squeezed toward 0. The probabilities are
  flattened toward the overall fraction of sick patients, and the model hardly dares to say anything.
- As `C` grows, $\|w\|$ grows, and the transition between the classes gets **sharper**.
- From about `C = 1` upward the picture barely changes: this data **overlaps**, so the loss has a
  natural minimum, and weak regularization doesn't matter much. (On separable data, the difference
  between `C = 100` and `C = 1000` would be dramatic, as in section 1.)

### 5.2 In the weights: coefficient paths

For the full Breast Cancer data (scaled), let's fit a model for many values of `C` and plot each of
the 30 weights against `C`. Each line is one feature: its **path** as the regularization changes.

In [ ]:
X_bc_s = StandardScaler().fit_transform(X_bc)
Cs_path = np.logspace(-3, 3, 25)
paths = np.array([LogisticRegression(C=C, max_iter=10000).fit(X_bc_s, y_bc).coef_[0] for C in Cs_path])

plt.figure(figsize=(9, 5))
plt.plot(Cs_path, paths, linewidth=1)
plt.xscale("log")
plt.axhline(0, color="k", linewidth=0.8)
plt.xlabel("C (log scale):  strong regularization  <-->  weak regularization")
plt.ylabel("weight")
plt.title("Coefficient paths: the 30 Breast Cancer weights as C changes")
plt.show()

Reading from right to left (increasing regularization), all the weights **shrink toward 0**. At the
far left they're all almost 0: the model barely uses any feature. From right to left, the weights don't
all shrink at the same speed: the features that matter most hold out longest.

### Exercise 5.A

At `C = 0.01` (strong regularization), which feature has the **largest** weight (in absolute value)?
Fit the model, and store the feature's **name** in `strongest_at_001`. (The names are in `bc_names`.)

In [ ]:
# YOUR CODE HERE
strongest_at_001 = None
print(strongest_at_001)
check("5.A", strongest_at_001)

### 5.3 In performance: underfitting and overfitting

Back to the small Breast Cancer problem from section 2 (40 training patients). For each `C`, we train,
and measure accuracy on the training data and on **other** data.

**How to choose `C`, the right way.** `C` is a choice we make. If we chose it by looking at the
**test** set, the test set would stop being "unseen", and its result would be too optimistic: the same
"peeking" problem as in 02. So we **split the training data again**:

- **training part:** fit the model;
- **validation part:** compare different values of `C`;
- **test set:** touched **once**, at the very end, for the chosen `C` only.

And we repeat the training/validation split several times and average, so that one lucky split
doesn't decide. (We'll use this "several splits" idea again in notebook 04.)

In [ ]:
Cs = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

# a fixed test set, set aside and NOT used below
X_rest, X_test, y_rest, y_test = train_test_split(X_bc, y_bc, test_size=0.3, stratify=y_bc, random_state=0)

train_acc = {C: [] for C in Cs}
val_acc = {C: [] for C in Cs}
for seed in range(10):
    # 40 patients for training, the rest (of X_rest) for validation
    X_tr, X_val, y_tr, y_val = train_test_split(X_rest, y_rest, train_size=40, stratify=y_rest, random_state=seed)
    sc = StandardScaler().fit(X_tr)
    X_tr, X_val = sc.transform(X_tr), sc.transform(X_val)
    for C in Cs:
        m = LogisticRegression(C=C, max_iter=10000).fit(X_tr, y_tr)
        train_acc[C].append(m.score(X_tr, y_tr))
        val_acc[C].append(m.score(X_val, y_val))

mean_train = [np.mean(train_acc[C]) for C in Cs]
mean_val = [np.mean(val_acc[C]) for C in Cs]
plt.figure(figsize=(8, 4.5))
plt.plot(Cs, mean_train, "o-", label="training accuracy")
plt.plot(Cs, mean_val, "o-", label="validation accuracy")
plt.xscale("log"); plt.xlabel("C (log scale)"); plt.ylabel("accuracy (average of 10 splits)")
plt.legend(); plt.title("Breast Cancer, 40 training patients")
plt.show()
for C, tr, va in zip(Cs, mean_train, mean_val):
    print(f"C = {C:<7}  train = {tr:.3f}   validation = {va:.3f}")

This is the classic picture:

- **Left (small `C`, too much regularization): underfitting.** Even the training accuracy is low: the
  model is too restricted to learn the pattern.
- **Right (large `C`, too little regularization): overfitting.** The training accuracy reaches 100%:
  the model fits the 40 training patients perfectly. The validation accuracy doesn't follow: there's a
  **gap** between the two curves, the signature of overfitting. (Here the validation accuracy stays
  roughly flat on the right. In the Ionosphere exercise in section 7 you'll see it actually **drop**.)
- **The sweet spot** is where the validation accuracy is highest.

Finally, and **only** now, we train with the chosen `C` and check it **once** on the test set:

In [ ]:
best_C = Cs[int(np.argmax(mean_val))]
X_tr, _, y_tr, _ = train_test_split(X_rest, y_rest, train_size=40, stratify=y_rest, random_state=0)
sc = StandardScaler().fit(X_tr)
final = LogisticRegression(C=best_C, max_iter=10000).fit(sc.transform(X_tr), y_tr)
print("chosen C:", best_C)
print("test accuracy of the final model:", round(final.score(sc.transform(X_test), y_test), 3))

The test accuracy of this single final model is lower than the average validation accuracy. A model
trained on only **40** patients depends a lot on *which* 40 it got, so one model's result can easily
be a few points above or below the average. That's exactly why we averaged over 10 splits when
**choosing** `C`, rather than trusting one split.

## 6. The third reason to scale

Back in 02 we promised a third reason to scale the features. Here it is. The penalty
$\frac{\lambda}{2}(w_1^2 + \dots + w_n^2)$ treats **all weights equally**. But the size of a weight
depends on the units of its feature (01b, exercise 5.B). A feature measured in small numbers needs a
**large** weight to have an effect, and so it gets penalized **much more** than a feature measured in
large numbers, for no good reason.

Let's see it happen. In the 5-feature data from 01b (all features on the same scale), we'll "measure"
feature `x1`, the most important one, in different units: we divide it by 100.

In [ ]:
rng = np.random.default_rng(5)
w_true = np.array([2.0, -1.5, 1.0, 0.0, 0.5])
X5 = rng.normal(size=(1000, 5))
y5 = (rng.uniform(size=1000) < 1 / (1 + np.exp(-(X5 @ w_true - 0.5)))).astype(int)
X5_units = X5.copy()
X5_units[:, 0] /= 100                     # the same information, in "different units"

X5_tr, X5_te, X5u_tr, X5u_te, y5_tr, y5_te = train_test_split(X5, X5_units, y5, test_size=0.3, random_state=0)
for C in [1, 0.01]:
    m_same = LogisticRegression(C=C).fit(X5_tr, y5_tr)
    m_units = LogisticRegression(C=C).fit(X5u_tr, y5_tr)
    print(f"C = {C}:")
    print(f"   same scale:     weights {m_same.coef_[0].round(3)}   test accuracy {m_same.score(X5_te, y5_te):.3f}")
    print(f"   x1 divided:     weights {m_units.coef_[0].round(3)}   test accuracy {m_units.score(X5u_te, y5_te):.3f}")

**Exercise 6.A (in words).** Look at the weight of `x1` in the "divided" model. To have the same
effect as before, it would need to be about 100 times larger than in the "same scale" model. What
happened to it instead, and what did that do to the test accuracy?

**Your answer:** *(write here)*

## 7. Exercise: choose `C` for the Ionosphere data

The **Ionosphere** dataset (you'll meet it again in notebook 04): radar signals, 33 numeric
measurements each, and the task is to classify each return as "good" (1) or "bad" (0). There are only
229 examples, so, with 33 features, regularization matters.

Follow the procedure of section 5.3:

1. Set aside a test set (30%, stratified, `random_state=0`).
2. For each $C \in \{0.001, 0.01, 0.1, 1, 10, 100, 1000\}$, average the **validation** accuracy over
   10 random training/validation splits of the remaining data (validation = 30% of it). Remember to
   scale inside each split!
3. Plot training and validation accuracy against `C`.
4. Choose the best `C`, train on all the non-test data with it, and report the **test** accuracy
   **once**.
5. In words: is there overfitting? Underfitting? How big is the difference between the best `C` and
   sklearn's default `C=1`?

In [ ]:
iono = pd.read_csv("../data/ionosphere.csv").drop(columns=["a2"])     # a2 is always 0: no information
y_io = (iono["class"] == "g").astype(int).values
X_io = iono.drop(columns=["class"]).values.astype(float)
print(X_io.shape, "  fraction 'good':", y_io.mean().round(3))

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Optional: a puzzle from the experiments

In 03a's optional challenge we found that the rescaled tanh model with weights $w$ is **the same** as a
logistic model with weights $2w$. Now add L2 regularization with the same $\lambda$ to both. Are they
still the same model?

**Your answer:** *(write here)*

## Recap

- **Separable data:** the log-loss has no minimum. Gradient descent sends $\|w\|$ to infinity, and the
  model becomes 100% sure of everything.
- **Many features, few examples:** the model memorizes the training data. That's **overfitting**:
  perfect training accuracy, worse test accuracy.
- **L2 regularization:**
  - add $\frac{\lambda}{2}\|w\|^2$ to the loss, without penalizing $b$;
  - the gradient gets $+\lambda w$ ("weight decay");
  - larger $\lambda$ gives smaller weights and softer probabilities.
- **sklearn's `C`** is the **inverse** strength: $\lambda = \frac{1}{C N}$.
  - Small `C` is strong regularization; the default is `C=1`.
  - `C=1e10` is practically no regularization.
- **Too much regularization underfits, too little overfits.** Choose `C` by **validation** accuracy,
  averaged over several splits, and use the test set only **once**, at the end.
- **Scale before regularizing**, so the penalty treats all features fairly.
- **Next (03c):** practice set 2. Then **04**: polynomial features, where regularization becomes
  essential.

## References and further reading

- Google Machine Learning Crash Course, *Logistic regression*,
  [Loss and regularization](https://developers.google.com/machine-learning/crash-course/logistic-regression/loss-regularization):
  why logistic regression needs regularization, in a few paragraphs.
- MLU-Explain, [The Bias-Variance Tradeoff](https://mlu-explain.github.io/bias-variance/): underfitting
  vs. overfitting, visually and interactively.
- MLU-Explain, [Train, Test, and Validation Sets](https://mlu-explain.github.io/train-test-validation/):
  why we choose `C` on a validation set.
- scikit-learn:
  [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
  (the parameter `C`), and the user guide section on
  [logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression),
  which shows the exact formula sklearn minimizes.